# Notebook 05 — Trend Analysis (Mann-Kendall)

**Purpose:** Apply the Mann-Kendall non-parametric trend test to each district's annual fire count time series (2015–2023) to detect statistically significant increasing or decreasing burning trends.

**Input:** `data/processed/fire_stats.csv`  
**Output:** `data/processed/fire_trends.csv`

In [12]:
import pandas as pd
import pymannkendall as mk

# Load fire statistics 
stats = pd.read_csv('Data/Processed/fire_stats.csv')
print(f"Shape: {stats.shape}")
print(f"Districts: {stats['district'].nunique()}")
print(f"Years    : {sorted(stats['year'].unique())}")


Shape: (386, 8)
Districts: 43
Years    : [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [13]:
# Mann-Kendall trend test per district

# The Mann-Kendall test is a non-parametric rank-based test that detects
# monotonic trends in a time series without assuming normality.
#
# Outputs per district:
#   trend     : 'increasing', 'decreasing', or 'no trend'
#   tau       : Kendall's tau (-1 to +1), strength of the trend
#   p_value   : statistical significance (significant if p < 0.05)
#   slope     : Sen's slope — estimated change in fire count per year
#   significant: True if p < 0.05
results = []

for district in stats['district'].unique():
    sub    = stats[stats['district'] == district].sort_values('year')
    series = sub['fire_count'].values

    # Need at least 4 time points for a meaningful test
    if len(series) < 4:
        print(f"  ⚠  Skipping {district} — only {len(series)} years of data")
        continue

    res = mk.original_test(series)
    results.append({
        'district'   : district,
        'trend'      : res.trend,
        'tau'        : round(res.Tau,   3),
        'p_value'    : round(res.p,     4),
        'slope'      : round(res.slope, 1),
        'significant': res.p < 0.05,
    })

trend_df = pd.DataFrame(results)
print(f"\nTrend results computed for {len(trend_df)} districts")
print(trend_df.head(8))



Trend results computed for 43 districts
    district       trend    tau  p_value  slope  significant
0     Ambala    no trend  0.389   0.1753   70.8        False
1    Bhiwani    no trend  0.222   0.4655   12.6        False
2  Faridabad  increasing  0.556   0.0476    9.0         True
3  Fatehabad    no trend -0.444   0.1179 -251.2        False
4   Gurugram    no trend  0.333   0.2515    4.5        False
5      Hisar    no trend  0.278   0.3481   69.7        False
6    Jhajjar    no trend  0.333   0.2515   15.2        False
7       Jind    no trend  0.333   0.2515  116.6        False


In [14]:
# Summary of trend directions
print("Trend distribution:")
print(trend_df['trend'].value_counts().to_string())
print()
print("Statistically significant (p < 0.05):", trend_df['significant'].sum())
print()
print("Top 5 districts by Sen slope (fastest growing burning):")
print(
    trend_df[trend_df['significant']]
    .nlargest(5, 'slope')[['district', 'trend', 'slope', 'p_value']]
    .to_string(index=False)
)


Trend distribution:
trend
no trend      41
increasing     1
decreasing     1

Statistically significant (p < 0.05): 2

Top 5 districts by Sen slope (fastest growing burning):
 district      trend  slope  p_value
Faridabad increasing    9.0   0.0476
    Sirsa decreasing -239.2   0.0286


In [15]:
# Save trend results
trend_df.to_csv('data/processed/fire_trends.csv', index=False)
print("Saved: data/processed/fire_trends.csv")
print(f"   Rows    : {len(trend_df)}")
print(f"   Columns : {trend_df.columns.tolist()}")


✅ Saved: data/processed/fire_trends.csv
   Rows    : 43
   Columns : ['district', 'trend', 'tau', 'p_value', 'slope', 'significant']
